## 01-Rerank-重排序

### 1. 是什么？
```
在多路召回之后，对召回的文档和跟用户问题进行重新打分、排序、过滤，将跟用户更相关的文档排到最前面，然后送给 LLM 生成答案
```

### 2. 要解决什么问题？
```
① 文档相似度很高，但是跟用户问题不相关
② 跟问题相关性比较高的文档排在很后面
```

### 3. 基本思路
RAG 重排放在什么位置
```
向量召回 Top-30  ─┐
                 ├─ Fusion(RRF) → 候选 20–30 条 → Rerank → Top 3–5 → LLM
BM25 召回 Top-30 ┘
```

也可以跳过 Fusion，直接把多路候选拼接去重后送 Rerank：
```
向量 Top-30 + BM25 Top-30 → 去重 → Rerank → Top 3–5
```

典型方法
```
① BGE-Reranker
② Cohere Rerank
```

### 4. 代码实例

#### 初始化重排模型

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

import dashscope
from langchain_community.document_compressors import DashScopeRerank
# 重排模型有：BGE-Reranker, Cohere Rerank等
reranker = DashScopeRerank(
    client = dashscope.TextReRank,
    model="gte-rerank-v2",                                 # 问题和文档---语义的深入理解
    top_n=3,
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),

)

#### 重排序API
- rerank API：重排序结果（含相关性分数）
- compress_documents API：重排序后返回的文档列表

In [4]:
from langchain_core.documents import Document

query = "孕妇感冒了怎么办"

# 文档列表按相关性从高到低设计，便于验证重排序模型是否能正确识别并排序
documents = [
    # ★★★★★ 高度相关：直接回答孕妇感冒该怎么处理
    Document(
        page_content="孕妇感冒后应多喝温水、多休息，如果发烧超过38.5度，可以在医生指导下使用对乙酰氨基酚，这是孕期相对安全的退烧药。",
        metadata={"source": "孕期感冒用药指南"},
    ),
    Document(
        page_content="孕期感冒首选物理方式缓解：用温水擦浴降温、生理盐水洗鼻缓解鼻塞、多饮温开水、保证充足睡眠，避免自行用药。",
        metadata={"source": "孕妇感冒护理"},
    ),
    # ★★★ 中度相关：提到了孕妇和感冒，但不是直接的操作建议
    Document(
        page_content="感冒了可以吃感康，但是孕妇禁用。感康中含有金刚烷胺等成分，可能对胎儿造成不良影响。",
        metadata={"source": "感康说明书"},
    ),
    Document(
        page_content="感冒了可以咨询专业医生，根据具体症状和体征获得个性化的诊断和治疗方案。",
        metadata={"source": "专业建议"},
    ),
    # ★★ 低度相关：讲感冒但不涉及孕妇
    Document(
        page_content="999感冒灵含有对乙酰氨基酚、马来酸氯苯那敏和咖啡因等成分，可缓解普通感冒引起的头痛、发热、鼻塞等症状。",
        metadata={"source": "999感冒灵"},
    ),
    Document(
        page_content="高血压患者感冒时应避免使用含伪麻黄碱的复方感冒药，以免引起血压升高，建议选择单一成分药物。",
        metadata={"source": "高血压患者"},
    ),
    # ★ 无关：医疗相关但与感冒/孕妇完全无关
    Document(
        page_content="糖尿病患者应控制碳水化合物摄入量，每日主食定量，优先选择低GI食物如燕麦、荞麦，并定期监测空腹及餐后血糖。",
        metadata={"source": "糖尿病饮食管理"},
    ),
]

print("=" * 70)
print(f"查询问题: {query}")
print(f"文档总数: {len(documents)}, 重排序返回 Top-{reranker.top_n}")
print("=" * 70)

# 重排模型自带的API，重新对文本进行排序
print("\n[rerank API] 重排序结果（含相关性分数）:")
print("-" * 70)
scores = reranker.rerank(documents, query) # documents上下文列表   query 是指用户的问题
print(scores)
for i, score in enumerate(scores):
    doc = documents[score["index"]]
    print(f"  Rank {i+1} | 分数: {score['relevance_score']:.4f} | 来源: {doc.metadata['source']}")
    print(f"          | 内容: {doc.page_content[:60]}...")
    print()

# 封装的重排序的API
print("\n[compress_documents API] 重排序后返回的文档列表:")
print("-" * 70)
compressed = reranker.compress_documents(documents, query)
for i, doc in enumerate(compressed):
    print(f"  Rank {i+1} | 来源: {doc.metadata['source']}")
    print(f"          | 内容: {doc.page_content[:80]}...")
    print()


查询问题: 孕妇感冒了怎么办
文档总数: 7, 重排序返回 Top-3

[rerank API] 重排序结果（含相关性分数）:
----------------------------------------------------------------------
[{'index': 1, 'relevance_score': 0.5381604741414563}, {'index': 0, 'relevance_score': 0.5343339149046874}, {'index': 3, 'relevance_score': 0.19036964020519254}]
  Rank 1 | 分数: 0.5382 | 来源: 孕妇感冒护理
          | 内容: 孕期感冒首选物理方式缓解：用温水擦浴降温、生理盐水洗鼻缓解鼻塞、多饮温开水、保证充足睡眠，避免自行用药。...

  Rank 2 | 分数: 0.5343 | 来源: 孕期感冒用药指南
          | 内容: 孕妇感冒后应多喝温水、多休息，如果发烧超过38.5度，可以在医生指导下使用对乙酰氨基酚，这是孕期相对安全的退烧药。...

  Rank 3 | 分数: 0.1904 | 来源: 专业建议
          | 内容: 感冒了可以咨询专业医生，根据具体症状和体征获得个性化的诊断和治疗方案。...


[compress_documents API] 重排序后返回的文档列表:
----------------------------------------------------------------------
  Rank 1 | 来源: 孕妇感冒护理
          | 内容: 孕期感冒首选物理方式缓解：用温水擦浴降温、生理盐水洗鼻缓解鼻塞、多饮温开水、保证充足睡眠，避免自行用药。...

  Rank 2 | 来源: 孕期感冒用药指南
          | 内容: 孕妇感冒后应多喝温水、多休息，如果发烧超过38.5度，可以在医生指导下使用对乙酰氨基酚，这是孕期相对安全的退烧药。...

  Rank 3 | 来源: 专业建议
          | 内容: 感冒了可以咨询专业医生，根据具体症状和体征获得个性化的诊断和治疗方案。...

